# 01 · Prognosis is not treatment effect

**Synthetic data.** Part 1 uses a toy cohort made in this notebook. Part 2 uses the Kaggle CIBMTR synthetic data.

## Key to abbreviations

| Term | Full name | Meaning here |
|---|---|---|
| HCT | Hematopoietic cell transplantation | A transplant of blood-forming stem cells, for example a bone marrow transplant. |
| CIBMTR | Center for International Blood and Marrow Transplant Research | The research group that hosts the Kaggle competition. |
| EFS | Event-free survival | The outcome. An "event" is relapse, graft failure or death. |
| Conditioning | — | Chemotherapy and/or radiation before the transplant. It clears the old marrow. |
| MAC | Myeloablative conditioning | High-dose conditioning. Better disease control, but more toxic. |
| RIC | Reduced-intensity conditioning | Lower-dose conditioning. Less toxic, so clinicians give it to older and frailer patients. |
| DCA | Decision curve analysis | A method that shows if a model gives a better decision than "act for all" or "act for none". |
| p_t | Threshold probability | The risk above which you act. See Part 2. |
| GAN | Generative adversarial network | A type of model that makes synthetic data. SurvivalGAN is the GAN that made the Kaggle data. |
| Cox | Cox proportional hazards model | A standard regression model for survival data. A name, not an acronym. |

## The two questions

| Question | Type | Example |
|---|---|---|
| "How likely is an event for this patient?" | **Prognosis** | Risk of an EFS event by 12 months is 40%. |
| "Will this patient do better with treatment X than with treatment Y?" | **Treatment effect** | RIC instead of MAC changes the risk from 40% to 30%. |

A prognostic model answers the first question. Our models (Cox, gradient-boosted survival) are prognostic models.

The trap: the model has treatment columns as inputs, for example `conditioning_intensity`. So you can change the input from MAC to RIC and read a new risk. **This looks like an answer to the second question. It is not.**

## Why not

In observational data, clinicians choose the treatment, and they choose it **for a reason**:
- They give RIC (reduced-intensity conditioning) to older and frailer patients.
- These patients have a high risk with any treatment.

So the RIC group has a higher event rate, even if RIC is the better choice for them. This is **confounding by indication**: the reason for the treatment also changes the outcome.

A model can adjust for the reasons that are **recorded** (age, comorbidity). It cannot adjust for reasons the clinician saw but did not record. These reasons are hidden confounders. Prediction can still be good, because the model does not need to know *why* a risk is high. A treatment comparison needs to know why.

## Plan

1. A toy cohort where we **know the truth** for each patient: the risk with MAC and the risk with RIC.
2. Three ways to read the treatment effect: naive, model, truth. See where each one goes wrong.
3. Use the same model for two decisions: an act-or-not decision (DCA works) and a treatment choice (DCA does not apply).
4. The same pattern in the CIBMTR synthetic data, where we do not know the truth.

In [ ]:
from hct_equity import teaching
from hct_equity.config import load_config
from hct_equity.data import load_train
import dcurves

project = load_config()
settings = project.teaching_simulation
print(project.synthetic_note)

## Part 1 · A toy cohort with known truth

Each simulated patient has:
- `frail`: recorded. Frail patients have a higher risk.
- `severe`: aggressive disease. **The clinician sees it, but it is not in the data.** Severe disease has a higher risk.
- `ric`: the treatment the clinician chose. Frail patients usually get RIC. Severe disease usually gets MAC.

The truth, which is set in `config/project.json`:
- For **frail** patients, RIC **lowers** the risk, because MAC is too toxic for them.
- For **fit** patients, RIC **raises** the risk, because MAC controls the disease better.

The best choice is different for different patients. Each patient has both risks (`risk_mac`, `risk_ric`). Real data shows only one of them.

In [ ]:
cohort = teaching.simulate_cohort(settings)
cohort.groupby(["frail", "severe"])[["ric", "risk_mac", "risk_ric"]].mean().round(2)

Read the table: `ric` is the share of patients who got RIC. Frail patients usually get RIC. Severe disease rarely gets RIC.

### The naive comparison

Compare event rates between the patients who got RIC and the patients who got MAC.

In [ ]:
cohort.groupby("ric")["event"].mean().rename("event_rate").round(3)

RIC looks **worse**. But RIC patients are mostly frail, and frail patients have a high risk with any treatment. This comparison measures *who got RIC*, not *what RIC does*.

### The prognostic model

Fit a logistic regression on the recorded columns: `frail`, `ric` and their interaction. This is a good prognostic model. Now use it the "trap" way: set every patient to MAC, then to RIC, and compare the predicted risks.

In [ ]:
model = teaching.fit_prognostic(cohort, teaching.OBSERVED_FEATURES)
teaching.effect_by_frailty(cohort, model, teaching.OBSERVED_FEATURES).round(3)

Each column is (risk with RIC) − (risk with MAC). A negative value means RIC helps.

| Column | Meaning |
|---|---|
| `true_effect` | The truth, from the two potential outcomes. |
| `model_effect` | What the model says when you change the `ric` input. |
| `naive_effect` | The difference in event rates, within each frailty group. |

For **fit patients (frail = 0)**, the truth is that RIC **harms**. The model says RIC **helps**. The sign is wrong.

Why? Fit patients with severe disease usually get MAC, and severe disease has a high risk. So the MAC group contains the patients with the worst disease. The model cannot see `severe`, so it blames MAC for their events.

The model adjusted for `frail`, so the confounding here does **not** come from `frail`. It comes from `severe`, which is not recorded.

### What happens if we use the model to choose treatment?

Each row is a policy, applied to all patients. `event_risk` is the true average risk under that policy. We can compute it only because this is a simulation.

In [ ]:
teaching.compare_policies(cohort, model, teaching.OBSERVED_FEATURES).round(3)

The model's policy gives **RIC to everyone**. That is **worse than current practice**: the clinicians, who see `severe`, already choose better than the model.

### Check: record the hidden variable

If `severe` were in the data, the model could adjust for it.

In [ ]:
full_model = teaching.fit_prognostic(cohort, teaching.ALL_FEATURES)
display(teaching.effect_by_frailty(cohort, full_model, teaching.ALL_FEATURES).round(3))
teaching.compare_policies(cohort, full_model, teaching.ALL_FEATURES).round(3)

Now `model_effect` is close to `true_effect`, and the model's policy matches the best policy.

**Lesson:** a treatment effect from a model is correct only if **every** reason for the treatment choice that also affects the outcome is recorded. Real data never shows whether this is true. This condition has a name: **no unmeasured confounding**. It is an assumption, not something the data can test.

## Part 2 · What DCA can and cannot answer

DCA compares **act** against **do not act** at a threshold probability p_t. The threshold sets the exchange rate between harm and benefit:

> p_t / (1 − p_t) = harm of an unnecessary action ÷ benefit of a necessary action

Example decision: **extra surveillance after HCT**, for example more frequent visits. It has low harm, so the relevant thresholds are low.

For this decision, the model only needs to **rank patients by risk**. It does not need the reason for the risk. So the confounded model from Part 1 is still useful here.

In [ ]:
thresholds = project.teaching_dca_thresholds.values()
dca_result = teaching.surveillance_dca(cohort, model, teaching.OBSERVED_FEATURES, thresholds)
dcurves.plot_graphs(plot_df=dca_result, graph_type="net_benefit", y_limits=(-0.05, 0.7))

How to read the plot:
- **Treat none** is 0 at all thresholds.
- **Treat all** is high at low thresholds, because about 59% of the toy cohort has an event. If action is cheap and most patients have an event, acting for everyone is a strong default.
- At low thresholds, the model gives every patient a risk above the threshold, so its line **equals** treat all. The model adds value only where it flags some patients and not others.
- At higher thresholds, the model line is above both defaults. In that range, the model is useful for the surveillance decision.

The same model gave a harmful treatment policy in Part 1. **One model can be useful for one decision and harmful for another.**

The CIBMTR data has the same feature: 12-month risk is about 52%. So treat all is strong at low thresholds, and our models must beat it.

| Decision | What the model must get right | Can a prognostic model do it? | Tool to evaluate |
|---|---|---|---|
| Act or do not act (surveillance, referral, trial screening) | Who has a high risk | Yes | DCA |
| Choose between treatments (MAC or RIC, donor type) | How the risk **changes** with the treatment, for each patient | Only with no unmeasured confounding | Randomised trial, or causal methods (target-trial emulation, propensity methods). DCA of a treatment-benefit model is possible, but only after a causal estimate. |

A DCA curve has one model line and one **action**. "Treatment A instead of treatment B" is not one action that a risk threshold can trigger.

## Part 3 · The same pattern in the CIBMTR synthetic data

There is no truth here. We see only the outcome under the treatment that the patient got.

In [ ]:
example = project.treatment_example
train = load_train(project)
teaching.arm_case_mix(train, example.column, example.arms, example.case_mix, project.columns.event).round(2)

RIC patients have a higher event rate. They are also older and have more comorbidity. This is the same pattern as Part 1, so the naive comparison is confounded.

A model that adjusts for age and comorbidity removes only the **recorded** part. Clinicians also choose by disease status, by performance details and by patient wishes, and not all of this is in the data.

Also, this data is **synthetic** (SurvivalGAN). A treatment effect in it shows what the generator learned. It is not a clinical fact.

## What this means for the CIBMTR summary

1. We evaluate our models as **prognostic** models: discrimination, calibration by race group, and DCA for act-or-not decisions.
2. We do **not** say that any model shows which conditioning, donor or graft is better.
3. The limits section must state this. A treatment comparison needs a causal design.